In [1]:
import psi4
import pandas as pd
import os
import numpy as np
from lps_rscf import lps_solver

In [2]:
csv_file = 'sic_closed_shell_atoms.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> No existing file found. Starting fresh.


In [32]:
def sic_p(N):
    return (1 - (2 / N) ** (2 / 3))

def sic_p_absp(N):
    return (1 - (2 / N) ** (1 / 3))

def sic_x(N):
    return (1 - (2 / N) ** (1 / 3))

def sic_p_test(N):
    return (1 - (1.412 * (N) ** (- 1 / 3)))

In [33]:
psi4.core.set_output_file('output.dat', False)

ATOMS = {
    'He':  {'mult': 1,'N': 2}, 
    'Be': {'mult': 1,'N': 4},
    'Ne': {'mult': 1,'N': 10}, 
    # 'Mg': {'mult': 1,'N': 12},
    # 'Ar':  {'mult': 1,'N': 18}, 
    # 'Ca':  {'mult': 1,'N': 20},
    # 'Zn':  {'mult': 1,'N': 30}, 
    # 'Kr':  {'mult': 1,'N': 36}
}

METHOD = "TEST"
TP = ['LDA_K_TF', 1.0]
LAMBDA = 1.0
EXC = ['LDA_X', 0.0, 'LDA_C_VWN', 0.0]
FA = [True, 1.0]
DIIS = True
MAX_ITER = 5000
DAMPING = [0.9, 0.0, 0.001]
D_guess = None
verbose=False

psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

for atom in ATOMS:
    
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    TP = ['LDA_K_TF', sic_p_test(ATOMS[atom]['N'])]
    # EXC = ['LDA_X', sic_x(ATOMS[atom]['N']), 'LDA_C_VWN', 0.0]
    try:
        E, D, mu, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,MOL,DAMPING,FA,D_guess,DIIS,verbose)
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print('\nFinal SCF energy: %.4f Hartree' \
                '\nConverged in %.i iterations' % ( E, iterations))
            # row = {
            #     "Method": METHOD,
            #     "Atom": atom,
            #     "Basis": psi4.core.get_global_option("BASIS"),
            #     "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
            #     "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
            #     "Energy,Ha": round(E, 6),
            #     "ChemPot,Ha": round(mu, 6),
            #     "Iterations": iterations,
            #     "DIIS": DIIS,
            #     "Damp_Start": DAMPING[0],
            #     "Damp_End": DAMPING[1],
            #     "Damp_Cutoff": DAMPING[2]
            # }
            
            # df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    
    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue

Calculating He with TEST...

Final SCF energy: -3.2105 Hartree
Converged in 29 iterations
Calculating Be with TEST...

Final SCF energy: -16.5130 Hartree
Converged in 62 iterations
Calculating Ne with TEST...

Final SCF energy: -145.9526 Hartree
Converged in 100 iterations


In [25]:
df.to_csv(csv_file, index=False)
display(df)

,Method,Atom,Basis,Grid_Sph,Grid_Rad,"Energy,Ha","ChemPot,Ha",Iterations,DIIS,Damp_Start,Damp_End,Damp_Cutoff
0,W TF_SIC FA,He,UGBS_S,6,1000,-2.861680,-0.917956,39,True,0.9,0.0,0.001
1,W TF_SIC FA,Be,UGBS_S,6,1000,-12.706809,-0.788604,69,True,0.9,0.0,0.001
2,W TF_SIC FA,Ne,UGBS_S,6,1000,-106.209206,-0.586168,108,True,0.9,0.0,0.001
3,W TF_SIC FA,Mg,UGBS_S,6,1000,-163.896738,-0.550891,117,True,0.9,0.0,0.001
4,W TF_SIC FA,Ar,UGBS_S,6,1000,-433.010279,-0.484604,157,True,0.9,0.0,0.001
5,W TF_SIC FA,Ca,UGBS_S,6,1000,-557.937756,-0.470124,169,True,0.9,0.0,0.001
6,W TF_SIC FA,Zn,UGBS_S,6,1000,-1483.084038,-0.423649,738,True,0.9,0.0,0.001
7,W TF_SIC FA,Kr,UGBS_S,6,1000,-2303.202334,-0.406844,465,True,0.9,0.0,0.001
8,W TF_ABSP FA,He,UGBS_S,6,1000,-2.861680,-0.917956,39,True,0.9,0.0,0.001
9,W TF_ABSP FA,Be,UGBS_S,6,1000,-14.848813,-1.128632,68,True,0.9,0.0,0.001


In [15]:
ATOMS = {
    'He':  {'mult': 1}, 
    'Be': {'mult': 1},
    'Ne': {'mult': 1}, 
    'Mg': {'mult': 1},
    'Ar':  {'mult': 1}, 
    'Ca':  {'mult': 1},
    'Zn':  {'mult': 1}, 
    'Kr':  {'mult': 1}
}
psi4.core.set_output_file('output.dat', False)
psi4.set_options({'basis': 'UGBS',
                  'scf_type': 'PK'})
rhf_energies = {}
rhf_homos = {}
for atom in ATOMS:
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    E, wfn = psi4.energy('SCF', return_wfn=True)
    homo = wfn.epsilon_a().np[wfn.nalpha()-1]
    rhf_energies[atom] = round(E, 6)
    rhf_homos[atom] = round(homo, 6)
    print(f"RHF/UGBS {atom} Energy: {E:.4f} Hartree, IP: {homo:.4f}")

RHF/UGBS He Energy: -2.8617 Hartree, IP: -0.9180
RHF/UGBS Be Energy: -14.5730 Hartree, IP: -0.3093
RHF/UGBS Ne Energy: -128.5471 Hartree, IP: -0.8504
RHF/UGBS Mg Energy: -199.6146 Hartree, IP: -0.2530
RHF/UGBS Ar Energy: -526.8175 Hartree, IP: -0.5910
RHF/UGBS Ca Energy: -676.7582 Hartree, IP: -0.1955
RHF/UGBS Zn Energy: -1777.8481 Hartree, IP: -0.2925
RHF/UGBS Kr Energy: -2752.0549 Hartree, IP: -0.5242


In [26]:
atom_order = ['He', 'Be', 'Ne', 'Mg', 'Ar', 'Ca', 'Zn', 'Kr']
energy_table = df.pivot(index='Atom', columns='Method', values='Energy,Ha')
energy_table = energy_table.reindex(atom_order)
energy_table['RHF/UGBS'] = pd.Series(rhf_energies)
new_order = ['W TF_ABSP FA S_SIC', 'W TF_ABSP FA', 'W TF_SIC FA S_SIC', 'W TF_SIC FA', 'TFW FA', 'RHF/UGBS']
energy_table = energy_table[new_order]

In [27]:
reference = energy_table['RHF/UGBS']

res = {}
for method, energies in energy_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
energy_table = pd.concat([energy_table, stats_df], sort=False)
display(energy_table)

,W TF_ABSP FA S_SIC,W TF_ABSP FA,W TF_SIC FA S_SIC,W TF_SIC FA,TFW FA,RHF/UGBS
He,-2.861680,-2.861680,-2.861680,-2.861680,-1.539176,-2.861680
Be,-15.476608,-14.848813,-13.231946,-12.706809,-8.318459,-14.573023
Ne,-140.273208,-134.355735,-110.674139,-106.209206,-82.790100,-128.547083
Mg,-216.918540,-208.285764,-170.371989,-163.896738,-131.025068,-199.614621
Ar,-570.728362,-551.511528,-447.405194,-433.010279,-363.016580,-526.817486
Ca,-733.535493,-710.037497,-575.569982,-557.937756,-472.757119,-676.758154
Zn,-1924.063567,-1874.048719,-1521.065989,-1483.084038,-1302.039741,-1777.848060
Kr,-2966.293519,-2896.577700,-2356.535810,-2303.202334,-2049.433009,-2752.054860
MAE(Ha),61.380000,39.180000,110.170000,127.020000,208.520000,NaN
rMAE(%),7.090000,3.880000,12.070000,14.540000,34.080000,NaN


In [29]:
atom_order = ['He', 'Be', 'Ne', 'Mg', 'Ar', 'Ca', 'Zn', 'Kr']
mu_table = df.pivot(index='Atom', columns='Method', values='ChemPot,Ha')
mu_table = mu_table.reindex(atom_order)
mu_table['RHF/UGBS'] = pd.Series(rhf_homos)
new_order = ['W TF_ABSP FA S_SIC', 'W TF_ABSP FA', 'W TF_SIC FA S_SIC', 'W TF_SIC FA', 'TFW FA', 'RHF/UGBS']
mu_table = mu_table[new_order]

reference = mu_table['RHF/UGBS']

res = {}
for method, energies in mu_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
mu_table = pd.concat([mu_table, stats_df], sort=False)

display(mu_table)

,W TF_ABSP FA S_SIC,W TF_ABSP FA,W TF_SIC FA S_SIC,W TF_SIC FA,TFW FA,RHF/UGBS
He,-0.917956,-0.917956,-0.917956,-0.917956,-0.285221,-0.917956
Be,-1.254602,-1.128632,-0.882170,-0.788604,-0.319221,-0.309271
Ne,-1.333714,-1.071905,-0.746061,-0.586168,-0.333822,-0.850411
Mg,-1.294854,-1.018386,-0.717878,-0.550891,-0.333141,-0.253048
Ar,-1.180520,-0.885779,-0.663305,-0.484604,-0.329266,-0.590989
Ca,-1.148844,-0.851866,-0.651255,-0.470124,-0.327909,-0.195527
Zn,-1.033838,-0.733961,-0.612832,-0.423649,-0.322032,-0.292463
Kr,-0.988229,-0.688803,-0.599210,-0.406844,-0.319229,-0.524161
MAE(Ha),0.650000,0.420000,0.260000,0.210000,0.230000,NaN
rMAE(%),212.940000,145.170000,93.800000,66.180000,40.720000,NaN
